# RSNA DINOv3: cross-series teacher/student alignment → label fine-tuning

**Stage 1:** For the same non-gold knee study, choose two different MRI series in the same anatomical plane. A frozen pretrained DINOv3 teacher supplies the clean series representation. A student initialized from the same checkpoint receives the other series with mild intensity augmentation, pools its eight sampled slice features, and predicts the teacher representation using cosine loss. An additional own-series feature anchor limits drift from the pretrained representation. Only the student's final transformer block, final LayerNorm and temporary predictor train. The teacher never receives gradients or EMA updates. This is a fixed-teacher distillation experiment, not a reproduction of DINOv3 or BYOL pretraining.

Pairs are made at the **series/study level**, not between assumed corresponding slices. Matching global representations encourages sequence invariance; it does not establish geometric equivariance. An abnormality may be visible in only one sequence, so improvement is an empirical question. Different anatomical planes are deliberately not forced together in this first experiment.

**Stage 2:** Discard the temporary predictor, initialize every supervised fold from the aligned student backbone, and train the original 12-label study classifier. Use 58 gold studies and 1,750 report-labelled studies. Soft-label multiplier stays 0.5; uncertain/not-mentioned masks stay 0.2/0.0. The final backbone block remains trainable. Checkpoint selection uses gold validation BCE; five-fold OOF scores and ensemble submission are saved.

**Evaluation separation:** Alignment uses only the 1,750 non-gold studies, with a fixed 90/10 study-level alignment train/validation split. No gold images or labels and no test studies enter alignment. The same non-gold alignment checkpoint initializes all folds. Alignment validation loss selects the SSL checkpoint; gold folds select only supervised checkpoints. Compare with the existing supervised-only DINOv3 notebook on the same gold splits and label CSV. Set USE_ALIGNMENT=False for a baseline run with the same cache/seed settings; output names are separate.

**Setup:** Attach RSNA data, rsna_medgemma_modal27b_soft1750.csv containing exactly 1,750 rows, and local facebook/dinov3-vits16-pretrain-lvd1689m files. The exact DINO and label paths from your run are already set in configuration; edit only if you attach different versions. Model loading is local-only. Enable a Kaggle GPU; dependency installation needs Internet unless packages are already present.

Run all cells in order. The alignment feature-bank cell uses the frozen teacher once; the alignment training cell runs 3 epochs by default, followed by five supervised folds. This uses additional GPU time. The cache has a new version because it now stores per-image series IDs; older caches cannot be used. Download the aligned backbone, fold checkpoints, history and OOF CSVs from /kaggle/working. No automatic checkpoint resume or checkpoint reuse is performed.

Synthetic implementation checks are run during notebook development. Full pretrained RSNA/GPU training has not been run here, and a lower alignment loss alone does not demonstrate better abnormality detection.

References: https://huggingface.co/docs/transformers/v4.56.2/en/model_doc/dinov3 ; https://arxiv.org/abs/2006.07733


In [1]:
%pip install -q 'transformers==4.56.2' safetensors pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg opencv-python-headless scikit-learn


ERROR: Could not find a version that satisfies the requirement transformers==4.56.2 (from versions: none)
ERROR: No matching distribution found for transformers==4.56.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ThreadPoolExecutor
import time, random, gc
import numpy as np
import pandas as pd
import torch, cv2, pydicom
from transformers import AutoModel, AutoConfig, AutoImageProcessor
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark=True
LABEL_COLS=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
IMAGE_SIZE=224; CENTERS_PER_PLANE=8; IMAGE_MICROBATCH=2
NUM_DICOM_THREADS=8; NUM_WORKERS=2; N_FOLDS=5; EPOCHS=5
PSEUDO_LOSS_WEIGHT=.5; EXPECTED_PSEUDO_COUNT=1750
WEAK_LABEL_CSV=Path('/kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1750.csv')
LR_BACKBONE=1e-5; LR_HEAD=2e-4
DATA_CANDIDATES=[Path('/kaggle/input/rsna-knee-abnormality-detection'),Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
DATA_ROOT=next((p for p in DATA_CANDIDATES if p.exists()),None)
if DATA_ROOT is None: raise FileNotFoundError('Attach RSNA Knee Abnormality Detection competition data.')
TRAIN_SERIES_ROOT=DATA_ROOT/'train_series'; TEST_SERIES_ROOT=DATA_ROOT/'test_series'
OUT=Path('/kaggle/working'); CACHE=OUT/'rsna_2p5d_medgemma1750_series_v3_cache'; CACHE.mkdir(exist_ok=True)
DINO_MODEL_ID='facebook/dinov3-vits16-pretrain-lvd1689m'
LOCAL_DINO_DIR=Path('/kaggle/input/models/amritgopinath/dino-v3/transformers/default/2')
print('device',DEVICE)

# Additional cross-series alignment experiment. No calls to Modal.
USE_ALIGNMENT=True
SSL_EPOCHS=3; SSL_LR_BACKBONE=1e-5; SSL_LR_PREDICTOR=1e-4
SSL_ANCHOR_WEIGHT=.25; SSL_VALID_FRACTION=.1
SSL_SEED=123; SSL_RUN_TAG='dinov3_crossseries1750'
RUN_TAG='dinov3_crossseries1750_w05' if USE_ALIGNMENT else 'dinov3_baseline1750_w05'
SSL_CHECKPOINT=OUT/f'{SSL_RUN_TAG}_backbone.pt'


device cuda


In [3]:
# Strictly local model files: no model download or authentication on Kaggle.
DINO_SOURCE=Path(LOCAL_DINO_DIR)
for name in ('config.json','preprocessor_config.json'):
    if not (DINO_SOURCE/name).is_file():
        raise FileNotFoundError(f'Missing {name} in {DINO_SOURCE}. Set LOCAL_DINO_DIR to the actual model folder.')
if not ((DINO_SOURCE/'model.safetensors').is_file() or (DINO_SOURCE/'model.safetensors.index.json').is_file()):
    raise FileNotFoundError(f'Missing model weights in {DINO_SOURCE}')
DINO_CONFIG=AutoConfig.from_pretrained(DINO_SOURCE,local_files_only=True)
assert DINO_CONFIG.model_type=='dinov3_vit', 'Expected DINOv3 ViT weights'
processor=AutoImageProcessor.from_pretrained(DINO_SOURCE,local_files_only=True)
DINO_MEAN=processor.image_mean; DINO_STD=processor.image_std
print('Local DINOv3 source:',DINO_SOURCE)


Local DINOv3 source: /kaggle/input/models/amritgopinath/dino-v3/transformers/default/2


In [4]:
train=pd.read_csv(DATA_ROOT/'train.csv',dtype={'StudyInstanceUID':str})
train_series=pd.read_csv(DATA_ROOT/'train_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
test_series=pd.read_csv(DATA_ROOT/'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})
sample_submission=pd.read_csv(DATA_ROOT/'sample_submission.csv',dtype={'StudyInstanceUID':str})
assert list(sample_submission.columns)==['StudyInstanceUID']+LABEL_COLS
gold=train[train[LABEL_COLS].notna().all(axis=1)].copy().reset_index(drop=True)
gold[LABEL_COLS]=gold[LABEL_COLS].apply(pd.to_numeric)
assert len(gold)==58, f'Expected 58 gold studies; got {len(gold)}'
assert gold.StudyInstanceUID.is_unique
print('gold studies',len(gold),'test studies',len(sample_submission))

# Load only the exact export name, never recursively scan attached inputs.
import hashlib
CSV_NAME='rsna_medgemma_modal27b_soft1750.csv'
if WEAK_LABEL_CSV is not None:
    csv_paths=[Path(WEAK_LABEL_CSV)]
else:
    root=Path('/kaggle/input')
    csv_paths=sorted(set([p for pattern in (f'*/{CSV_NAME}',f'datasets/*/*/{CSV_NAME}')
                         for p in root.glob(pattern)] + ([OUT/CSV_NAME] if (OUT/CSV_NAME).is_file() else [])))
if len(csv_paths)!=1 or not csv_paths[0].is_file():
    raise FileNotFoundError(f'Set WEAK_LABEL_CSV to the 1750-row {CSV_NAME}; found {csv_paths}')
print('Reading weak labels from',csv_paths[0])
pseudo=pd.read_csv(csv_paths[0],dtype={'StudyInstanceUID':str,'report_sha256':str})
assert len(pseudo)==EXPECTED_PSEUDO_COUNT and pseudo.StudyInstanceUID.is_unique, f'Expected {EXPECTED_PSEUDO_COUNT} unique rows; got {len(pseudo)}'
required={'StudyInstanceUID','report_sha256','model','prompt_version',*LABEL_COLS,
          *('weight__'+x for x in LABEL_COLS),*('status__'+x for x in LABEL_COLS)}
assert required.issubset(pseudo.columns), f'Missing CSV columns: {required-set(pseudo.columns)}'
assert 'Report' in train.columns and train.StudyInstanceUID.is_unique
assert set(pseudo.StudyInstanceUID).isdisjoint(set(gold.StudyInstanceUID)), 'Gold study in weak labels'
assert set(pseudo.StudyInstanceUID).issubset(set(train.StudyInstanceUID)), 'Unexpected study ID'
assert pseudo.model.eq('google/medgemma-27b-text-it').all()
assert pseudo.prompt_version.eq('medgemma-knee-v1-modal-text-v1').all()
report_lookup=train.set_index('StudyInstanceUID')['Report']
for row in pseudo[['StudyInstanceUID','report_sha256']].itertuples(index=False):
    digest=hashlib.sha256(str(report_lookup.loc[row.StudyInstanceUID]).encode('utf-8')).hexdigest()
    if row.report_sha256!=digest:
        raise ValueError(f'Report hash mismatch for study {row.StudyInstanceUID}')
soft_map={'positive':(.95,1.),'negative':(.05,1.),'uncertain':(.5,.2),'not_mentioned':(.5,0.)}
for col in LABEL_COLS:
    values=pd.to_numeric(pseudo[col],errors='raise')
    weights=pd.to_numeric(pseudo['weight__'+col],errors='raise')
    expected=pseudo['status__'+col].map(soft_map)
    if expected.isna().any(): raise ValueError(f'Invalid status for {col}')
    assert np.allclose(values.to_numpy(),expected.map(lambda v:v[0]).to_numpy())
    assert np.allclose(weights.to_numpy(),expected.map(lambda v:v[1]).to_numpy())
    pseudo[col]=values.astype(float)
    pseudo['weight__'+col]=weights.astype(float)
pseudo=pseudo.sample(frac=1,random_state=SEED).reset_index(drop=True)
print('report-only pseudo rows',len(pseudo),'usable targets',int(pseudo[['weight__'+x for x in LABEL_COLS]].gt(0).sum().sum()))


gold studies 58 test studies 3
Reading weak labels from /kaggle/input/datasets/amritgopinath/medgemma-27b-labels/rsna_medgemma_modal27b_soft1750.csv
report-only pseudo rows 1750 usable targets 14617


In [5]:
def backbone_tail(net):
    # Transformers releases expose the ViT blocks at different nesting levels.
    for path in ('layer', 'model.layer', 'encoder.layer'):
        blocks=net
        for part in path.split('.'):
            blocks=getattr(blocks,part,None)
            if blocks is None: break
        if isinstance(blocks,(nn.ModuleList,nn.Sequential)) and len(blocks):
            break
    else:
        raise RuntimeError(f'Unsupported DINOv3 layout: {type(net).__name__}; children={list(dict(net.named_children()))}')
    norm=next((getattr(net,name,None) for name in ('norm','layernorm')
               if isinstance(getattr(net,name,None),nn.Module)),None)
    if norm is None: raise RuntimeError(f'Missing final normalization in {type(net).__name__}')
    return blocks[-1],norm

def train_backbone_tail(net):
    net.eval()
    for module in backbone_tail(net): module.train()

def make_backbone(load_pretrained=True):
    net=(AutoModel.from_pretrained(DINO_SOURCE,local_files_only=True) if load_pretrained
         else AutoModel.from_config(DINO_CONFIG))
    for p in net.parameters(): p.requires_grad_(False)
    for module in backbone_tail(net):
        for p in module.parameters(): p.requires_grad_(True)
    return net

class StudyDINOv3(nn.Module):
    def __init__(self,load_pretrained=True):
        super().__init__()
        self.backbone=make_backbone(load_pretrained)
        dim=self.backbone.config.hidden_size
        self.head=nn.Sequential(nn.LayerNorm(dim),nn.Dropout(.15),nn.Linear(dim,len(LABEL_COLS)))
        self.plane_weight=nn.Parameter(torch.zeros(4,len(LABEL_COLS)))
        self.register_buffer('mean',torch.tensor(DINO_MEAN).view(1,3,1,1))
        self.register_buffer('std',torch.tensor(DINO_STD).view(1,3,1,1))

    def train_backbone_tail(self):
        train_backbone_tail(self.backbone)

    def forward(self,images,planes):
        slice_logits=[]
        for chunk in images.split(IMAGE_MICROBATCH):
            x=(chunk.to(dtype=self.mean.dtype)/255-self.mean)/self.std
            ctx=torch.autocast('cuda',dtype=torch.float16) if self.training and images.is_cuda else nullcontext()
            with ctx: slice_logits.append(self.head(self.backbone(pixel_values=x).last_hidden_state[:,0]))
        scores=torch.cat(slice_logits,0)
        pooled=[]; available=[]
        for plane in range(4):
            part=scores[planes==plane]
            available.append(len(part)>0)
            pooled.append(part.topk(min(2,len(part)),dim=0).values.mean(0) if len(part) else torch.zeros_like(scores[0]))
        mask=torch.tensor(available,device=images.device)
        weights=self.plane_weight.masked_fill(~mask[:,None],-1e4).softmax(0)
        return (weights*torch.stack(pooled)).sum(0)

# Check checkpoint compatibility before the long DICOM caching stage.
_probe_backbone=make_backbone(load_pretrained=True)
print('Backbone class:',type(_probe_backbone).__name__, 'trainable parameters:',sum(p.numel() for p in _probe_backbone.parameters() if p.requires_grad))
del _probe_backbone
print("DINOv3 checkpoint loaded; last block and final LayerNorm will fine-tune")


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Backbone class: DINOv3ViTModel trainable parameters: 1775616
DINOv3 checkpoint loaded; last block and final LayerNorm will fine-tune


In [6]:
def plane_of(row):
    p=str(row.get('Anatomical_Plane','')).strip().upper()
    if p.startswith('SAG'): return 'SAGITTAL'
    if p.startswith('COR'): return 'CORONAL'
    if p.startswith('AX'): return 'AXIAL'
    return 'UNKNOWN'

def select_series(df, ids):
    df=df.copy()
    df['StudyInstanceUID']=df.StudyInstanceUID.astype(str)
    df['SeriesInstanceUID']=df.SeriesInstanceUID.astype(str)
    df=df[df.StudyInstanceUID.isin(set(map(str,ids)))].drop_duplicates(['StudyInstanceUID','SeriesInstanceUID'])
    df['plane_norm']=df.apply(plane_of,axis=1)
    for col in ('Fluid_Sensitive','Fat_Suppression'):
        if col not in df: df[col]='UNKNOWN'
        df[col]=df[col].fillna('UNKNOWN').astype(str).str.upper().str.strip()
    rows=[]
    for sid,group in df.groupby('StudyInstanceUID',sort=True):
        group=group.sort_values('SeriesInstanceUID')
        picks=[]
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            if len(candidates): picks.append(candidates.iloc[0])
        for plane in ('SAGITTAL','CORONAL','AXIAL'):
            candidates=group[group.plane_norm.eq(plane)]
            used=[p for p in picks if p.plane_norm==plane]
            if len(candidates)<2 or not used: continue
            first=used[0]
            rest=candidates[candidates.SeriesInstanceUID.ne(first.SeriesInstanceUID)]
            diverse=rest[(rest.Fluid_Sensitive.ne(first.Fluid_Sensitive)) | (rest.Fat_Suppression.ne(first.Fat_Suppression))]
            if len(diverse): picks.append(diverse.iloc[0])
            elif len(rest): picks.append(rest.iloc[0])
        if not picks and len(group): picks.append(group.iloc[0])
        rows.extend([{'StudyInstanceUID':sid,'SeriesInstanceUID':str(r.SeriesInstanceUID),'plane_norm':r.plane_norm} for r in picks])
    return pd.DataFrame(rows,columns=['StudyInstanceUID','SeriesInstanceUID','plane_norm'])

train_picks=select_series(train_series,pd.concat([gold.StudyInstanceUID,pseudo.StudyInstanceUID],ignore_index=True))
test_picks=select_series(test_series,sample_submission.StudyInstanceUID)
train_picks.to_csv(OUT/'dinov3_crossseries_series_selection.csv',index=False)
assert set(gold.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Gold study has no metadata series'
assert set(pseudo.StudyInstanceUID).issubset(set(train_picks.StudyInstanceUID)), 'Pseudo study has no metadata series'
print('gold series',len(train_picks),'test series',len(test_picks))


gold series 8968 test series 14


In [7]:
def _read_one(path):
    try:
        d=pydicom.dcmread(str(path),force=True)
        x=np.squeeze(d.pixel_array)
        if x.ndim!=2: return None
        x=x.astype(np.float32)*float(getattr(d,'RescaleSlope',1))+float(getattr(d,'RescaleIntercept',0))
        if str(getattr(d,'PhotometricInterpretation',''))=='MONOCHROME1': x=x.max()-x
        o=np.asarray(getattr(d,'ImageOrientationPatient',[]),float)
        pos=np.asarray(getattr(d,'ImagePositionPatient',[]),float)
        g=np.nan
        if len(o)==6 and len(pos)==3:
            n=np.cross(o[:3],o[3:]); norm=np.linalg.norm(n)
            if norm>0: g=float(pos @ (n/norm))
        try: inst=float(d.InstanceNumber)
        except Exception: inst=np.nan
        return (path.name,g,inst,x)
    except Exception: return None

def resize_pad(x):
    h,w=x.shape; scale=IMAGE_SIZE/max(h,w)
    nh,nw=max(1,round(h*scale)),max(1,round(w*scale))
    z=cv2.resize(x,(nw,nh),interpolation=cv2.INTER_AREA if scale<1 else cv2.INTER_LINEAR)
    out=np.zeros((IMAGE_SIZE,IMAGE_SIZE),np.uint8)
    top=(IMAGE_SIZE-nh)//2; left=(IMAGE_SIZE-nw)//2
    out[top:top+nh,left:left+nw]=np.rint(z*255).astype(np.uint8)
    return out

def load_plane(study,series,root):
    directory=root/str(study)/str(series)
    paths=sorted(p for p in directory.iterdir() if p.is_file() and not p.name.startswith('.')) if directory.is_dir() else []
    if not paths: return None
    with ThreadPoolExecutor(max_workers=NUM_DICOM_THREADS) as pool: rows=[r for r in pool.map(_read_one,paths) if r is not None]
    if not rows: return None
    key=1 if all(np.isfinite(r[1]) for r in rows) else 2 if all(np.isfinite(r[2]) for r in rows) else 0
    rows.sort(key=lambda r:(r[key],r[0]) if key else (r[0],))
    samples=np.concatenate([r[3][::8,::8].ravel() for r in rows])
    samples=samples[np.isfinite(samples)]
    if not len(samples): return None
    lo,hi=np.percentile(samples,[1,99])
    n=len(rows); centers=np.rint(np.linspace(.05*(n-1),.95*(n-1),CENTERS_PER_PLANE)).astype(int)
    needed=sorted(set(np.clip(np.r_[centers-1,centers,centers+1],0,n-1)))
    maps={}
    for i in needed:
        x=np.nan_to_num(rows[i][3],nan=lo,posinf=hi,neginf=lo)
        x=(np.clip(x,lo,hi)-lo)/(hi-lo) if hi>lo else np.zeros_like(x)
        maps[i]=resize_pad(x.astype(np.float32))
    return np.stack([np.stack([maps[max(0,i-1)],maps[i],maps[min(n-1,i+1)]],0) for i in centers],0)

def cache_path(study,root):
    return CACHE/('train_' if root==TRAIN_SERIES_ROOT else 'test_')/f'{study}.npz'

def cache_study(study,picks,root):
    path=cache_path(study,root)
    if path.exists(): return path
    blocks=[]; planes=[]; series_index=[]
    group=picks[picks.StudyInstanceUID.eq(str(study))]
    for series_no,row in enumerate(group.itertuples(index=False)):
        try: block=load_plane(study,row.SeriesInstanceUID,root)
        except Exception: block=None
        if block is None: continue
        series_index.extend([series_no]*len(block))
        blocks.append(block); planes.extend([('SAGITTAL','CORONAL','AXIAL').index(row.plane_norm) if row.plane_norm in ('SAGITTAL','CORONAL','AXIAL') else 3]*len(block))
    if not blocks: return None
    path.parent.mkdir(parents=True,exist_ok=True)
    np.savez(path,images=np.concatenate(blocks,0),planes=np.asarray(planes,np.int8),series_index=np.asarray(series_index,np.int16))
    return path

def warm_cache(ids,picks,root):
    t=time.time(); failures=[]
    for i,sid in enumerate(map(str,ids),1):
        if cache_study(sid,picks,root) is None: failures.append(sid)
        if i%100==0 or i==len(ids): print(f'cached {i}/{len(ids)} in {(time.time()-t)/60:.1f} min; failed={len(failures)}')
    return failures


gold_failures=warm_cache(gold.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
assert not gold_failures, f'Gold studies failed DICOM decode: {gold_failures}'
pseudo_failures=warm_cache(pseudo.StudyInstanceUID.tolist(),train_picks,TRAIN_SERIES_ROOT)
if pseudo_failures: print('Dropped undecodable pseudo studies:',len(pseudo_failures))
pseudo=pseudo.loc[~pseudo.StudyInstanceUID.isin(pseudo_failures)].reset_index(drop=True)
assert len(pseudo)>0, 'No pseudo studies decoded.'


cached 58/58 in 1.1 min; failed=0
cached 100/1750 in 2.1 min; failed=0
cached 200/1750 in 4.1 min; failed=0
cached 300/1750 in 6.0 min; failed=0
cached 400/1750 in 7.8 min; failed=0
cached 500/1750 in 9.7 min; failed=0
cached 600/1750 in 11.7 min; failed=0
cached 700/1750 in 13.6 min; failed=0
cached 800/1750 in 15.4 min; failed=0
cached 900/1750 in 17.0 min; failed=0
cached 1000/1750 in 18.9 min; failed=0
cached 1100/1750 in 20.8 min; failed=0
cached 1200/1750 in 22.6 min; failed=0
cached 1300/1750 in 24.3 min; failed=0
cached 1400/1750 in 25.9 min; failed=0
cached 1500/1750 in 27.6 min; failed=0
cached 1600/1750 in 29.4 min; failed=0
cached 1700/1750 in 31.1 min; failed=0
cached 1750/1750 in 31.9 min; failed=0


In [8]:
from itertools import combinations
from torch.nn import functional as F

def same_plane_pairs(series_index,planes):
    # A series has one plane. Unknown-plane pairs are excluded.
    series_plane={int(k):int(planes[series_index==k][0]) for k in np.unique(series_index)}
    return [(a,b) for a,b in combinations(sorted(series_plane),2)
            if series_plane[a]==series_plane[b] and series_plane[a]<3]

@torch.no_grad()
def mild_intensity(images,generator):
    # Keep geometry and neighboring-channel order; perturb intensities together.
    x=images.float()/255.
    gain=float(torch.empty(1).uniform_(.9,1.1,generator=generator))
    bias=float(torch.empty(1).uniform_(-.03,.03,generator=generator))
    return ((x*gain+bias).clamp(0,1)*255).round().to(torch.uint8)

def series_feature(net,images):
    mean=torch.tensor(DINO_MEAN,device=images.device).view(1,3,1,1)
    std=torch.tensor(DINO_STD,device=images.device).view(1,3,1,1)
    features=[]
    for chunk in images.split(IMAGE_MICROBATCH):
        x=(chunk.float()/255.-mean)/std
        ctx=torch.autocast('cuda',dtype=torch.float16) if images.is_cuda else nullcontext()
        with ctx:
            features.append(net(pixel_values=x).last_hidden_state[:,0].float())
    # Pool eight slice-center CLS features into one series feature.
    return torch.cat(features).mean(0)

class SeriesPredictor(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.net=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,dim),nn.GELU(),nn.Linear(dim,dim))
    def forward(self,x):return self.net(x)

def alignment_loss(z_student,target_other,target_own,predictor):
    # Detached fixed teacher targets. No status/label data enters this loss.
    cross=1-F.cosine_similarity(predictor(z_student).float(),target_other.detach().float(),dim=-1)
    anchor=1-F.cosine_similarity(z_student.float(),target_own.detach().float(),dim=-1)
    return cross+SSL_ANCHOR_WEIGHT*anchor,cross,anchor

ssl_pairs={}; teacher_bank={}
if USE_ALIGNMENT:
    assert set(pseudo.StudyInstanceUID).isdisjoint(set(gold.StudyInstanceUID))
    for sid in pseudo.StudyInstanceUID.astype(str):
        with np.load(cache_path(sid,TRAIN_SERIES_ROOT)) as d:
            pairs=same_plane_pairs(d['series_index'],d['planes'])
        if pairs:ssl_pairs[sid]=pairs
    if len(ssl_pairs)<10:
        raise RuntimeError(f'Only {len(ssl_pairs)} studies have distinct same-plane series; need at least 10 for this experiment.')
    ssl_ids=sorted(ssl_pairs)
    rng=np.random.default_rng(SSL_SEED);rng.shuffle(ssl_ids)
    n_valid=max(1,round(len(ssl_ids)*SSL_VALID_FRACTION))
    ssl_valid_ids=ssl_ids[:n_valid];ssl_train_ids=ssl_ids[n_valid:]
    assert set(ssl_train_ids).isdisjoint(ssl_valid_ids)
    assert set(ssl_ids).isdisjoint(set(gold.StudyInstanceUID.astype(str)))
    pd.DataFrame({'StudyInstanceUID':ssl_ids,'split':['validation']*n_valid+['train']*(len(ssl_ids)-n_valid)}).to_csv(OUT/f'{SSL_RUN_TAG}_split.csv',index=False)
    teacher=make_backbone(load_pretrained=True).to(DEVICE).eval()
    for p in teacher.parameters():p.requires_grad_(False)
    for i,sid in enumerate(ssl_ids,1):
        with np.load(cache_path(sid,TRAIN_SERIES_ROOT)) as d:
            images=d['images'];indices=d['series_index']
            series_ids=sorted({k for pair in ssl_pairs[sid] for k in pair})
            with torch.no_grad():
                for k in series_ids:
                    feature=series_feature(teacher,torch.from_numpy(images[indices==k].copy()).to(DEVICE))
                    teacher_bank[(sid,k)]=feature.detach().cpu()
        if i%100==0 or i==len(ssl_ids):print('Frozen teacher features',i,'/',len(ssl_ids))
    bank=torch.stack(list(teacher_bank.values()))
    assert torch.isfinite(bank).all()
    print('Paired studies:',len(ssl_ids),'alignment train:',len(ssl_train_ids),'validation:',len(ssl_valid_ids))
    print('Teacher mean feature std:',float(bank.std(dim=0).mean()))
    torch.save(teacher_bank,OUT/f'{SSL_RUN_TAG}_teacher_features.pt')
    del teacher;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
else:
    print('Alignment disabled: supervised baseline will use original DINOv3 weights.')


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Frozen teacher features 100 / 1750
Frozen teacher features 200 / 1750
Frozen teacher features 300 / 1750
Frozen teacher features 400 / 1750
Frozen teacher features 500 / 1750
Frozen teacher features 600 / 1750
Frozen teacher features 700 / 1750
Frozen teacher features 800 / 1750
Frozen teacher features 900 / 1750
Frozen teacher features 1000 / 1750
Frozen teacher features 1100 / 1750
Frozen teacher features 1200 / 1750
Frozen teacher features 1300 / 1750
Frozen teacher features 1400 / 1750
Frozen teacher features 1500 / 1750
Frozen teacher features 1600 / 1750
Frozen teacher features 1700 / 1750
Frozen teacher features 1750 / 1750
Paired studies: 1750 alignment train: 1575 validation: 175
Teacher mean feature std: 0.18157558143138885


In [9]:
def load_ssl_series(sid,series_no):
    with np.load(cache_path(sid,TRAIN_SERIES_ROOT)) as d:
        images=d['images'][d['series_index']==series_no].copy()
    assert len(images)>0
    return torch.from_numpy(images)

@torch.no_grad()
def validate_alignment(student,predictor,ids):
    student.eval();predictor.eval();losses=[];crosses=[];anchors=[];representations=[]
    # Fixed pairs and both directions make epoch comparisons consistent.
    for sid in ids:
        for a,b in ssl_pairs[sid]:
            for own,other in ((a,b),(b,a)):
                z=series_feature(student,load_ssl_series(sid,own).to(DEVICE))
                loss,cross,anchor=alignment_loss(z,teacher_bank[(sid,other)].to(DEVICE),teacher_bank[(sid,own)].to(DEVICE),predictor)
                losses.append(float(loss));crosses.append(float(cross));anchors.append(float(anchor))
                representations.append(F.normalize(z,dim=-1).cpu())
    spread=float(torch.stack(representations).std(dim=0).mean())
    return float(np.mean(losses)),float(np.mean(crosses)),float(np.mean(anchors)),spread

if USE_ALIGNMENT:
    random.seed(SSL_SEED);np.random.seed(SSL_SEED);torch.manual_seed(SSL_SEED)
    student=make_backbone(load_pretrained=True).to(DEVICE)
    predictor=SeriesPredictor(student.config.hidden_size).to(DEVICE)
    optimizer=torch.optim.AdamW([
        {'params':[p for p in student.parameters() if p.requires_grad],'lr':SSL_LR_BACKBONE},
        {'params':predictor.parameters(),'lr':SSL_LR_PREDICTOR}],weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
    base=validate_alignment(student,predictor,ssl_valid_ids)
    history=[dict(epoch=0,train_loss=np.nan,val_loss=base[0],val_cross=base[1],val_anchor=base[2],normalized_feature_std=base[3])]
    best=base[0];best_epoch=0
    def save_aligned(epoch):
        torch.save({'backbone':{k:v.detach().cpu() for k,v in student.state_dict().items()},
                    'source_model':DINO_MODEL_ID,'ssl_epoch':epoch,
                    'ssl_train_ids':ssl_train_ids,'ssl_valid_ids':ssl_valid_ids,
                    'anchor_weight':SSL_ANCHOR_WEIGHT},SSL_CHECKPOINT)
    save_aligned(0)
    print('Initial alignment validation:',base)
    for epoch in range(1,SSL_EPOCHS+1):
        started=time.time();train_backbone_tail(student);predictor.train()
        rng=np.random.default_rng(SSL_SEED+epoch);order=rng.permutation(ssl_train_ids)
        generator=torch.Generator().manual_seed(SSL_SEED+epoch)
        losses=[];grads_checked=False
        for i,sid in enumerate(order,1):
            pair=ssl_pairs[sid][int(rng.integers(len(ssl_pairs[sid])))]
            own,other=pair if rng.random()<.5 else pair[::-1]
            images=mild_intensity(load_ssl_series(sid,own),generator).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            z=series_feature(student,images)
            loss,cross,anchor=alignment_loss(z,teacher_bank[(sid,other)].to(DEVICE),teacher_bank[(sid,own)].to(DEVICE),predictor)
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite alignment loss')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            if not grads_checked:
                grads=[p.grad for p in backbone_tail(student)[0].parameters() if p.requires_grad and p.grad is not None]
                if not grads or not any(torch.isfinite(g).all() and g.abs().sum()>0 for g in grads):
                    raise RuntimeError('No valid gradient reaches the final student block.')
                grads_checked=True
            torch.nn.utils.clip_grad_norm_([p for group in optimizer.param_groups for p in group['params']],1.)
            scaler.step(optimizer);scaler.update();losses.append(float(loss.detach()))
            if i%100==0 or i==len(order):print(f'Alignment epoch {epoch}: {i}/{len(order)} studies')
        val=validate_alignment(student,predictor,ssl_valid_ids)
        if not np.isfinite(val).all():raise RuntimeError('Nonfinite alignment validation')
        if val[0]<best:
            best=val[0];best_epoch=epoch;save_aligned(epoch)
        history.append(dict(epoch=epoch,train_loss=float(np.mean(losses)),val_loss=val[0],val_cross=val[1],val_anchor=val[2],normalized_feature_std=val[3]))
        pd.DataFrame(history).to_csv(OUT/f'{SSL_RUN_TAG}_history.csv',index=False)
        print(f'Alignment epoch {epoch}/{SSL_EPOCHS}: val={val[0]:.4f}, cross={val[1]:.4f}, anchor={val[2]:.4f}, spread={val[3]:.6f}, best={best_epoch}, minutes={(time.time()-started)/60:.1f}')
    print('Aligned backbone:',SSL_CHECKPOINT,'best epoch:',best_epoch)
    print('Check OOF abnormality AUC after label training; alignment loss alone is not a task metric.')
    del student,predictor,optimizer,scaler;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Initial alignment validation: (0.9344050661165114, 0.9344050676850547, 3.747075621844732e-09, 0.023187441751360893)
Alignment epoch 1: 100/1575 studies
Alignment epoch 1: 200/1575 studies
Alignment epoch 1: 300/1575 studies
Alignment epoch 1: 400/1575 studies
Alignment epoch 1: 500/1575 studies
Alignment epoch 1: 600/1575 studies
Alignment epoch 1: 700/1575 studies
Alignment epoch 1: 800/1575 studies
Alignment epoch 1: 900/1575 studies
Alignment epoch 1: 1000/1575 studies
Alignment epoch 1: 1100/1575 studies
Alignment epoch 1: 1200/1575 studies
Alignment epoch 1: 1300/1575 studies
Alignment epoch 1: 1400/1575 studies
Alignment epoch 1: 1500/1575 studies
Alignment epoch 1: 1575/1575 studies
Alignment epoch 1/3: val=0.0618, cross=0.0617, anchor=0.0004, spread=0.023260, best=1, minutes=2.8
Alignment epoch 2: 100/1575 studies
Alignment epoch 2: 200/1575 studies
Alignment epoch 2: 300/1575 studies
Alignment epoch 2: 400/1575 studies
Alignment epoch 2: 500/1575 studies
Alignment epoch 2: 600

In [10]:
class BagDS(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        with np.load(cache_path(str(row.StudyInstanceUID),TRAIN_SERIES_ROOT)) as d:
            x=d['images'].copy(); plane=d['planes'].astype(np.int64)
        return torch.from_numpy(x),torch.from_numpy(plane),torch.tensor(row[LABEL_COLS].to_numpy(np.float32))

class PseudoBagDS(BagDS):
    def __getitem__(self,i):
        x,plane,y=super().__getitem__(i)
        row=self.frame.iloc[i]
        weight=torch.tensor(row[['weight__'+c for c in LABEL_COLS]].to_numpy(np.float32))
        return x,plane,y,weight

def train_fold(model,frame,validation,fold,checkpoint):
    gold_loader=DataLoader(BagDS(frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                      collate_fn=lambda b:b[0])
    print("trainable backbone parameters",sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
    optimizer=torch.optim.AdamW([{'params':[p for p in model.backbone.parameters() if p.requires_grad],'lr':LR_BACKBONE},
                                 {'params':list(model.head.parameters())+[model.plane_weight],'lr':LR_HEAD}],weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
    best_bce=np.inf; best_epoch=None; history=[]
    # The same pseudo order is shared by folds, but the model and gold split differ.
    order=np.random.default_rng(SEED).permutation(len(pseudo))
    weak_chunks=np.array_split(order,EPOCHS)
    for epoch in range(1,EPOCHS+1):
        model.train(); model.train_backbone_tail()
        losses=[]; start=time.time()
        weak_frame=pseudo.iloc[weak_chunks[epoch-1]]
        assert len(weak_frame)>0
        weak_loader=DataLoader(PseudoBagDS(weak_frame),batch_size=1,shuffle=True,num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0,
                          collate_fn=lambda b:b[0])
        gold_iter=iter(gold_loader); weak_iter=iter(weak_loader)
        schedule=np.array([0]*len(frame)+[1]*len(weak_frame))
        np.random.default_rng(SEED+fold*100+epoch).shuffle(schedule)
        for mode in schedule:
            if mode==0:
                x,plane,y=next(gold_iter); weights=None
            else:
                x,plane,y,weights=next(weak_iter)
            x=x.to(DEVICE,non_blocking=True); plane=plane.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits=model(x,plane).float()
            if mode==0:
                loss=nn.functional.binary_cross_entropy_with_logits(logits,y)
            else:
                weights=weights.to(DEVICE)
                raw=nn.functional.binary_cross_entropy_with_logits(logits,y,reduction='none')
                # Preserve uncertain/not-mentioned masking and don't inflate a study with few asserted findings.
                loss=PSEUDO_LOSS_WEIGHT*(raw*weights).sum()/weights.numel()
            if not torch.isfinite(loss): raise RuntimeError(f'Nonfinite loss in fold {fold}, epoch {epoch}')
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
        pred,failed=predict_ids(model,validation.StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
        if failed: raise RuntimeError(f'Fold {fold} validation decode failures: {failed}')
        aligned=validation[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
        y=validation[LABEL_COLS].to_numpy(np.float32)
        probs=aligned[LABEL_COLS].to_numpy(np.float32).clip(1e-6,1-1e-6)
        val_bce=nn.functional.binary_cross_entropy(torch.from_numpy(probs),torch.from_numpy(y)).item()
        aucs=[roc_auc_score(y[:,j],probs[:,j]) for j in range(len(LABEL_COLS)) if np.unique(y[:,j]).size==2]
        val_auc=float(np.mean(aucs)) if aucs else float('nan')
        if val_bce<best_bce-1e-6:
            best_bce=val_bce; best_epoch=epoch
            torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},checkpoint)
        history.append({'fold':fold,'epoch':epoch,'train_weighted_loss':float(np.mean(losses)),
                        'val_bce':val_bce,'val_macro_auc':val_auc,'n_auc_labels':len(aucs),
                        'best_so_far':best_epoch,'minutes':(time.time()-start)/60})
        print(f'fold {fold} epoch {epoch}/{EPOCHS}: weighted loss {np.mean(losses):.4f}, '
              f'val BCE {val_bce:.4f}, val AUC {val_auc:.4f} ({len(aucs)} labels), '
              f'best epoch {best_epoch}, time {(time.time()-start)/60:.1f}m')
    model.load_state_dict(torch.load(checkpoint,map_location='cpu',weights_only=True),strict=True)
    return best_epoch,best_bce,history

@torch.inference_mode()
def predict_ids(model,ids,picks,root):
    model.eval(); rows=[]; failed=[]
    for sid in map(str,ids):
        path=cache_path(sid,root)
        if not path.exists(): cache_study(sid,picks,root)
        if not path.exists(): failed.append(sid); continue
        with np.load(path) as d:
            x=torch.from_numpy(d['images'].copy()).to(DEVICE); plane=torch.from_numpy(d['planes'].astype(np.int64)).to(DEVICE)
        score=torch.sigmoid(model(x,plane).float()).cpu().numpy()
        rows.append([sid,*score.tolist()])
    return pd.DataFrame(rows,columns=['StudyInstanceUID']+LABEL_COLS),failed

folds=KFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
oof=np.full((len(gold),len(LABEL_COLS)),np.nan,np.float32)
fold_scores=[]; epoch_history=[]
for fold,(tr,va) in enumerate(folds.split(gold),1):
    random.seed(SEED+fold); np.random.seed(SEED+fold); torch.manual_seed(SEED+fold)
    print('\nFOLD',fold,'train',len(tr),'validation',len(va))
    model=StudyDINOv3(load_pretrained=True)
    if USE_ALIGNMENT:
        state=torch.load(SSL_CHECKPOINT,map_location='cpu',weights_only=True)
        assert state['source_model']==DINO_MODEL_ID
        model.backbone.load_state_dict(state['backbone'],strict=True)
    model=model.to(DEVICE)
    checkpoint=OUT/f'{RUN_TAG}_fold{fold}.pt'
    best_epoch,best_bce,history=train_fold(model,gold.iloc[tr],gold.iloc[va],fold,checkpoint)
    epoch_history.extend(history)
    pred,failed=predict_ids(model,gold.iloc[va].StudyInstanceUID,train_picks,TRAIN_SERIES_ROOT)
    assert not failed
    aligned=gold.iloc[va][['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',validate='one_to_one')
    oof[va]=aligned[LABEL_COLS].to_numpy(np.float32)
    fold_scores.append({'fold':fold,'n_train':len(tr),'n_validation':len(va),'checkpoint':str(checkpoint),'best_epoch':best_epoch,'best_val_bce':best_bce})
    print('saved',checkpoint)
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
assert np.isfinite(oof).all()
oof_df=gold[['StudyInstanceUID']+LABEL_COLS].copy()
for j,label in enumerate(LABEL_COLS): oof_df['pred__'+label]=oof[:,j]
oof_df.to_csv(OUT/f'{RUN_TAG}_oof_58.csv',index=False)
pd.DataFrame(fold_scores).to_csv(OUT/f'{RUN_TAG}_fold_manifest.csv',index=False)
pd.DataFrame(epoch_history).to_csv(OUT/f'{RUN_TAG}_epoch_history.csv',index=False)
metrics=[]
for j,label in enumerate(LABEL_COLS):
    y=gold[label].to_numpy(float)
    metrics.append({'label':label,'positives':int(y.sum()),'auc':roc_auc_score(y,oof[:,j]) if np.unique(y).size==2 else np.nan})
metrics=pd.DataFrame(metrics); metrics.to_csv(OUT/f'{RUN_TAG}_oof_auc.csv',index=False)
display(metrics); print('5-fold OOF macro AUC:',metrics.auc.mean())



FOLD 1 train 46 validation 12


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

trainable backbone parameters 1775616


/tmp/ipykernel_23/3645375545.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 1 epoch 1/5: weighted loss 0.2638, val BCE 0.5697, val AUC 0.5849 (10 labels), best epoch 1, time 2.3m
fold 1 epoch 2/5: weighted loss 0.2537, val BCE 0.5930, val AUC 0.5895 (10 labels), best epoch 1, time 2.3m
fold 1 epoch 3/5: weighted loss 0.2548, val BCE 0.6295, val AUC 0.5847 (10 labels), best epoch 1, time 2.3m
fold 1 epoch 4/5: weighted loss 0.2488, val BCE 0.6166, val AUC 0.5868 (10 labels), best epoch 1, time 2.3m
fold 1 epoch 5/5: weighted loss 0.2445, val BCE 0.6236, val AUC 0.5980 (10 labels), best epoch 1, time 2.3m
saved /kaggle/working/dinov3_crossseries1750_w05_fold1.pt

FOLD 2 train 46 validation 12


...

trainable backbone parameters 1775616


/tmp/ipykernel_23/3645375545.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')


fold 5 epoch 1/5: weighted loss 0.2641, val BCE 0.6803, val AUC 0.5569 (12 labels), best epoch 1, time 2.3m
fold 5 epoch 2/5: weighted loss 0.2534, val BCE 0.6648, val AUC 0.6320 (12 labels), best epoch 2, time 2.3m
fold 5 epoch 3/5: weighted loss 0.2568, val BCE 0.6791, val AUC 0.6708 (12 labels), best epoch 2, time 2.3m
fold 5 epoch 4/5: weighted loss 0.2485, val BCE 0.6748, val AUC 0.6518 (12 labels), best epoch 2, time 2.3m
fold 5 epoch 5/5: weighted loss 0.2451, val BCE 0.6533, val AUC 0.6575 (12 labels), best epoch 5, time 2.3m
saved /kaggle/working/dinov3_crossseries1750_w05_fold5.pt


,label,positives,auc
0,ACL,24,0.378676
1,MCL,9,0.598639
2,Medial Meniscus,26,0.594952
3,Lateral Meniscus,23,0.602484
4,Medial OA,15,0.728682
5,Lateral OA,11,0.837524
6,PF OA,21,0.662806
7,Effusion,35,0.783851
8,Synovitis,27,0.627240
9,Baker's,12,0.625000


5-fold OOF macro AUC: 0.6181004862659826


In [11]:
# The test DICOMs are decoded and cached once; the five fold checkpoints are used sequentially.
test_ids=sample_submission.StudyInstanceUID.astype(str).tolist()
test_failures=warm_cache(test_ids,test_picks,TEST_SERIES_ROOT)
print('test decode failures',len(test_failures))
ensemble=np.zeros((len(test_ids),len(LABEL_COLS)),np.float64)
for fold in range(1,N_FOLDS+1):
    model=StudyDINOv3(load_pretrained=False)
    state=torch.load(OUT/f'{RUN_TAG}_fold{fold}.pt',map_location='cpu',weights_only=True)
    model.load_state_dict(state,strict=True)
    model=model.to(DEVICE)
    if DEVICE.type=='cuda': model=model.half()  # half weights and half inputs, no autocast at inference
    pred,failed=predict_ids(model,test_ids,test_picks,TEST_SERIES_ROOT)
    aligned=sample_submission[['StudyInstanceUID']].merge(pred,on='StudyInstanceUID',how='left',validate='one_to_one')
    fallback=gold[LABEL_COLS].mean().to_numpy(float)
    for j,label in enumerate(LABEL_COLS): aligned[label]=aligned[label].fillna(float(fallback[j]))
    ensemble+=aligned[LABEL_COLS].to_numpy(float)/N_FOLDS
    print('fold',fold,'predicted',len(pred),'missing',len(failed))
    del model; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
submission=sample_submission[['StudyInstanceUID']].copy()
submission[LABEL_COLS]=ensemble.astype(np.float32)
assert list(submission.columns)==list(sample_submission.columns)
assert submission.StudyInstanceUID.is_unique and len(submission)==len(sample_submission)
assert np.isfinite(ensemble).all() and ensemble.min()>=0 and ensemble.max()<=1
submission.to_csv(OUT/'submission.csv',index=False)
print('wrote',OUT/'submission.csv','shape',submission.shape)
display(submission.head())


cached 3/3 in 0.1 min; failed=0
test decode failures 0
fold 1 predicted 3 missing 0
fold 2 predicted 3 missing 0
fold 3 predicted 3 missing 0
fold 4 predicted 3 missing 0
fold 5 predicted 3 missing 0
wrote /kaggle/working/submission.csv shape (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.264751,0.197508,0.482095,0.261892,0.521484,0.412639,0.599088,0.590422,0.544326,0.376618,0.488036,0.207990
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.241298,0.194915,0.440150,0.247623,0.476680,0.334963,0.545091,0.623955,0.528932,0.346381,0.486548,0.209755
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.250416,0.161562,0.422518,0.241357,0.402485,0.289540,0.518548,0.565486,0.484425,0.302815,0.486333,0.194514
